# Week 35, Day 5: Deep networks as encoding models

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w035/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A linear model of pixels

Write ridge_r2(F, lam): append a constant column to the features F, fit ridge weights on the training stimuli, predict the test stimuli and return the held-out R² for the global array resp.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(14)
P, N = 32, 3000
X = r.normal(size=(N, P))
g = np.array([-1, -0.5, 1, 2, 1, -0.5, -1.]); g /= np.linalg.norm(g)

def conv_relu(X, f):
    L = len(f)
    return np.maximum(0, np.stack([X[:, j:j + L] @ f for j in range(X.shape[1] - L + 1)], 1))

clean = conv_relu(X, g).sum(1)
sd = 0.5 * clean.std()
resp = clean + r.normal(0, sd, N)
tr = np.arange(N) < 2000; te = ~tr

def ridge_r2(F, lam=1.0):
    F = np.c_[F, np.ones(len(F))]
    A = F[tr]
    w = ...
    pred = F[te] @ w
    return 1 - np.sum((resp[te] - pred) ** 2) / np.sum((resp[te] - resp[te].mean()) ** 2)

r2_pix = ridge_r2(X)
print(f"held-out R squared from pixels: {r2_pix:.2f}")

In [ ]:
import numpy as np
def test_r2_fn():
    "ridge_r2 gives R squared of about 1 for a noise-free linear target"
    w = np.arange(P) / P
    saved = resp.copy()
    try:
        resp[:] = X @ w
        assert ridge_r2(X, lam=1e-6) > 0.999
    finally:
        resp[:] = saved

def test_pixels_poor():
    "pixels explain less than 30% of held-out variance"
    assert r2_pix < 0.3

test_r2_fn()
test_pixels_poor()
print("All checks passed")

### Exercise: Pixels, random features and a filter bank

Build the bank of 16 filters (a Gaussian window exp(-t²/6) times cos(f t + phase), t from -3 to 3, normalised), the full feature matrix F_bank (each filter's rectified output at every position, concatenated) and the pooled matrix F_pool (each filter's rectified output summed over positions), then compare all four held-out R² values.

In [ ]:
W = r.normal(size=(P, 300)) / np.sqrt(P)
F_rand = np.maximum(0, X @ W)
t = np.arange(7) - 3
bank = []
for f in (0.5, 0.8, 1.2, 1.6):
    for ph in (0, np.pi / 2, np.pi, 3 * np.pi / 2):
        b = np.exp(-t ** 2 / 6) * np.cos(f * t + ph)
        bank.append(b / np.linalg.norm(b))
F_bank = ...
F_pool = ...
r2s = {"pixels": r2_pix, "random": ridge_r2(F_rand), "bank": ridge_r2(F_bank), "pooled": ridge_r2(F_pool)}
print({m: round(float(v), 2) for m, v in r2s.items()})
plt.bar(list(r2s), list(r2s.values())); plt.ylabel("held-out R squared"); plt.show()

In [ ]:
import numpy as np
def test_order():
    "random features beat pixels, and the filter bank beats random features"
    assert r2s["random"] > r2s["pixels"] + 0.15
    assert r2s["bank"] > r2s["random"] + 0.15

def test_pooled():
    "the pooled bank does at least as well as the full bank with far fewer features"
    assert r2s["pooled"] > r2s["bank"] - 0.02 and F_pool.shape[1] == 16

test_order()
test_pooled()
print("All checks passed")

### Exercise: How good could any model be?

Simulate a second repeat, resp2 = clean + new noise of the same standard deviation. On the test stimuli, estimate the noise ceiling as the covariance of the two repeats divided by the variance of resp, then divide each model's R² by it.

In [ ]:
resp2 = clean + r.normal(0, sd, N)
a, b = resp[te], resp2[te]
ceiling = ...
frac = {m: v / ceiling for m, v in r2s.items()}
print(f"noise ceiling {ceiling:.2f}")
print({m: round(float(v), 2) for m, v in frac.items()}, "fraction of explainable variance")

In [ ]:
import numpy as np
def test_ceiling():
    "the estimated ceiling is within 0.05 of the true value, 1/(1 + 0.25) = 0.8"
    assert abs(ceiling - 0.8) < 0.05

def test_fraction():
    "the pooled bank captures more than 90% of the explainable variance"
    assert frac["pooled"] > 0.9

test_ceiling()
test_fraction()
print("All checks passed")